# 04 · Split and candidate pools

**Input** (from `02_cleaning`, in `data/processed/`): `petitions.parquet`, `gold_pairs.parquet`, `out_of_scope.parquet`.

**Output** (in `data/processed/`, not in git):

| File | Content |
|---|---|
| `splits.parquet` | one row per petition: `id` and `split` (train / val / test / excluded) |

Steps: 1 load → 2 split by decision time → 3 retrieval queries per split → 4 candidate pools → 5 out-of-scope petitions by period → 6 save.

The split dates and the pool rule are in `src/splits.py`, so the later notebooks and the final pipeline use exactly the same rules.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent                      # this notebook runs inside notebooks/
sys.path.insert(0, str(ROOT / "src"))
from preprocess import CLASSES
from splits import SPLITS, assign_split, CandidatePools

DATA = ROOT / "data" / "processed"

## 1. Load

In [2]:
petitions = pd.read_parquet(DATA / "petitions.parquet")
gold = pd.read_parquet(DATA / "gold_pairs.parquet")
out_of_scope = pd.read_parquet(DATA / "out_of_scope.parquet")

assert (len(petitions), len(gold), len(out_of_scope)) == (127_191, 61_132, 886)
print(f"{len(petitions):,} petitions, {len(gold):,} gold pairs, {len(out_of_scope):,} out-of-scope petitions")

127,191 petitions, 61,132 gold pairs, 886 out-of-scope petitions


## 2. Split by decision time

Locked rule 3. The decision time is the moment the moderator decided: `opened_at` if the petition was published,
`rejected_at` if it was rejected (made in 02). All times are in UTC.

| Split | Decision time | Used for |
|---|---|---|
| train | before 1 Jul 2025 | fitting the models |
| val | 1 Jul – 31 Dec 2025 | choosing settings (C, k1, …), calibration and the fusion model |
| test | 1 Jan – 31 Aug 2026 | the final evaluation, used **once** (notebook 09) |
| excluded | after 31 Aug 2026 | not used (locked rule): 1–25 Sep 2026 is only part of a month, right up to the snapshot date |

**Why by time and never at random:** the assistant will be used on *future* petitions. A random split would put petitions
from the same weeks — often near-copies of each other — into both train and test, so the scores would look better than
they will be in real use. A time split copies real use: learn from the past, test on what came later.

In [3]:
petitions["split"] = assign_split(petitions["decision_time"])

table = pd.crosstab(petitions["label"], petitions["split"]).reindex(index=CLASSES, columns=SPLITS)
table.loc["Total"] = table.sum()
table

split,train,val,test,excluded
label,,,,
published,35174,1855,3017,335
duplicate,41908,1161,1737,77
irrelevant,18513,278,505,51
no-action,15363,455,563,66
honours,2957,42,33,3
already-happening,2237,91,103,13
local,265,148,224,17
Total,116417,4030,6182,562


In [4]:
# The class x split table must match the locked table from Milestone 1 exactly.
LOCKED = pd.DataFrame({
    "train": [35_174, 41_908, 18_513, 15_363, 2_957, 2_237, 265],
    "val":   [1_855, 1_161, 278, 455, 42, 91, 148],
    "test":  [3_017, 1_737, 505, 563, 33, 103, 224],
}, index=CLASSES)
pd.testing.assert_frame_equal(table.loc[CLASSES, ["train", "val", "test"]], LOCKED, check_names=False)
assert table.loc["Total", "excluded"] == 562
print("The split table matches the locked table.")

# each split covers only its own dates
petitions.groupby("split")["decision_time"].agg(["min", "max"]).reindex(SPLITS)

The split table matches the locked table.


,min,max
split,,
train,2015-07-20 11:05:07.590+00:00,2025-06-30 15:32:53.472+00:00
val,2025-07-01 07:13:47.358+00:00,2025-12-31 11:11:38.828+00:00
test,2026-01-02 10:14:17.193+00:00,2026-08-28 16:00:04.925+00:00
excluded,2026-09-01 09:05:36.497+00:00,2026-09-25 16:55:20.794+00:00


**The class mix changes over time.** `local` only exists since July 2023, so it is 0.2% of train but almost 4% of
validation and test, and the share of published petitions is much higher in the recent periods. This matters later:
the models learn from the train mix but are tested on the newer mix. Macro-F1 gives every class the same weight,
and the calibration and the fusion model are fitted on validation, which is close to the test period.
The excluded weeks (1–25 Sep 2026) also have an unusual mix: 59.6% published and only 13.7% duplicates.

In [5]:
shares = pd.crosstab(petitions["label"], petitions["split"], normalize="columns").reindex(index=CLASSES, columns=SPLITS)
(100 * shares).round(1)

split,train,val,test,excluded
label,,,,
published,30.2,46.0,48.8,59.6
duplicate,36.0,28.8,28.1,13.7
irrelevant,15.9,6.9,8.2,9.1
no-action,13.2,11.3,9.1,11.7
honours,2.5,1.0,0.5,0.5
already-happening,1.9,2.3,1.7,2.3
local,0.2,3.7,3.6,3.0


## 3. Retrieval queries per split

A duplicate with at least one valid gold link (from 02) is a **retrieval query**: the system should find the
petition(s) it duplicates. A query belongs to the split of its own decision time.

In [6]:
gold["split"] = gold["query_id"].map(petitions.set_index("id")["split"])
assert gold["split"].notna().all()
links_per_query = gold.groupby(["split", "query_id"]).size()

queries = pd.DataFrame({
    "queries": links_per_query.groupby(level="split").size(),
    "with several valid links": (links_per_query > 1).groupby(level="split").sum(),
}).reindex(SPLITS)
assert queries["queries"].tolist() == [41_744, 1_155, 1_728, 77]     # locked: train 41,744 · val 1,155 · test 1,728
queries

,queries,with several valid links
split,,
train,41744,6477
val,1155,102
test,1728,247
excluded,77,7


## 4. Candidate pools

For retrieval, a petition is compared only with the petitions a moderator could have linked **at the moment of the
decision** (locked rule 5). For a petition decided at time *t*, the **open pool** is every published petition that

- was opened by *t* + 48 h (a moderator sometimes links a petition that is about to be published), and
- was not closed before *t* (`closed_at` is empty, or on or after *t*),
- except the petition itself.

The **stress pool** drops the "not closed" condition: every published petition opened by *t* + 48 h. It is a harder
test, used only in the final evaluation.

Two details matter:

1. **Members are chosen by label (`published`), not by "has an opening time".** 49 petitions were published and
   later rejected (39 in this table; the other 10 are out of scope). They have an opening time but mostly no closing
   time, so a rule based on `opened_at` would put them in every later pool, although a rejected petition can never be
   a correct answer.
2. **The petition itself is never in its own pool.** A published petition is decided at its opening time, so it would
   always find itself, with similarity 1.0. That would be a fake signal in the similarity features (notebook 08).

The pools are not stored. `CandidatePools` in `src/splits.py` keeps the opening and closing times of the published
petitions and builds the pool for any time *t* when it is needed.

In [7]:
pools = CandidatePools(petitions)
print(f"{len(pools.ids):,} published petitions can be in a pool")

# detail 1: published-then-rejected petitions have an opening time, but they are never pool members
published_then_rejected = petitions[petitions["opened_at"].notna() & petitions["label"].ne("published")]
assert len(published_then_rejected) == 39
assert not np.isin(published_then_rejected["id"], pools.ids).any()
print(f"{len(published_then_rejected)} in-scope petitions were published and later rejected "
      f"({published_then_rejected['closed_at'].isna().sum()} have no closing time): none is in a pool")

40,381 published petitions can be in a pool
39 in-scope petitions were published and later rejected (38 have no closing time): none is in a pool


In [8]:
# detail 2: without the self rule, a published petition would be in its own pool
published_vt = petitions[petitions["split"].isin(["val", "test"]) & petitions["label"].eq("published")]
t, q = published_vt["decision_time"].iloc[0], published_vt["id"].iloc[0]
print(f"Petition #{q}: in its own pool without the self rule: {q in pools.pool_ids(t)}; "
      f"with it: {q in pools.pool_ids(t, q)}")

assert not any(q in pools.pool_ids(t, q) for t, q in zip(published_vt["decision_time"], published_vt["id"]))
print(f"Checked all {len(published_vt):,} published val/test petitions: none is in its own pool")

Petition #730194: in its own pool without the self rule: True; with it: False


Checked all 4,872 published val/test petitions: none is in its own pool


**Pool sizes.** The open pool is computed for every petition; the stress pool only for validation and test,
because it is only used in the final evaluation.

In [9]:
petitions["pool"] = pools.sizes(petitions["decision_time"], petitions["id"])
vt = petitions["split"].isin(["val", "test"])
petitions.loc[vt, "stress_pool"] = pools.sizes(petitions.loc[vt, "decision_time"], petitions.loc[vt, "id"], stress=True)

test = petitions[petitions["split"] == "test"]
assert abs(test["pool"].median() - 2_256) <= 0.01 * 2_256              # locked: median pool at test time ≈ 2,256
assert abs(test["stress_pool"].median() - 38_500) <= 0.01 * 38_500     # locked: stress pool ≈ 38.5k

is_query = petitions["id"].isin(gold["query_id"])
print("Median open pool of the retrieval queries:",
      petitions[is_query].groupby("split")["pool"].median().reindex(["val", "test"]).to_dict())
print("Petitions whose pool has fewer than 5 members (the top-5 similarity features in 08 must handle them):")
print(petitions.loc[petitions["pool"] < 5, ["id", "source", "label", "created_at", "decision_time", "pool"]].to_string(index=False))

petitions.groupby("split")[["pool", "stress_pool"]].agg(["median", "min", "max"]).reindex(SPLITS)

Median open pool of the retrieval queries: {'val': 1897.0, 'test': 2232.0}
Petitions whose pool has fewer than 5 members (the top-5 similarity features in 08 must handle them):
    id  source     label                    created_at                 decision_time  pool
313513 2019-24 duplicate 2020-04-02 21:35:14.095+00:00 2024-10-19 03:19:26.316+00:00     0


pool             stress_pool                  
          median   min   max      median      min      max
split                                                     
train     1911.0     0  3760         NaN      NaN      NaN
val       1929.0  1638  2080     36142.0  35235.0  37030.0
test      2257.0  1804  2503     38517.0  37029.0  40046.0
excluded  2336.0  2292  2397         NaN      NaN      NaN

The only empty pool belongs to **#313513**, the petition from the Milestone-1 audit that was rejected 4.5 years
after it was created. It was rejected as a duplicate on 19 Oct 2024, between two Parliaments: all 2019-24 petitions
closed when Parliament was dissolved (29 May 2024, 23:01 UTC), and the first petition of the new Parliament opened
on 31 Oct 2024. So no petition was open at that moment. It is in train, so it does not affect any evaluation.

**Are the correct answers in the pool?** If no gold target of a query is in its pool, no retrieval method can
find it. So the share of queries with at least one gold target in the pool is the best Recall@k any method can reach.

In [10]:
pairs = gold.merge(petitions[["id", "decision_time"]], left_on="query_id", right_on="id")
assert len(pairs) == len(gold)
pairs["in_open_pool"] = [target in pools.pool_ids(t, q)
                         for q, t, target in zip(pairs["query_id"], pairs["decision_time"], pairs["target_id"])]
pairs["in_stress_pool"] = [target in pools.pool_ids(t, q, stress=True)
                           for q, t, target in zip(pairs["query_id"], pairs["decision_time"], pairs["target_id"])]

assert pairs["in_stress_pool"].all()            # rule 4 keeps only targets opened by t + 48 h
assert pairs["in_open_pool"].mean() >= 0.998    # locked: 99.9% of gold targets are open at t
print(f"Gold targets in the open pool: {pairs['in_open_pool'].mean():.2%} "
      f"({(~pairs['in_open_pool']).sum()} were closed before the decision); in the stress pool: {pairs['in_stress_pool'].mean():.0%}")

best_recall = pairs.groupby(["split", "query_id"])["in_open_pool"].any().groupby(level="split").mean()
(100 * best_recall.reindex(SPLITS)).round(2).rename("queries with a gold target in the open pool (%)")

Gold targets in the open pool: 99.87% (79 were closed before the decision); in the stress pool: 100%


split
train       99.88
val         99.83
test        99.94
excluded    98.70
Name: queries with a gold target in the open pool (%), dtype: float64

## 5. Out-of-scope petitions by period

The 886 out-of-scope petitions (fake-name, operational, foi) are never used to train or tune anything. In the final
evaluation the model is run on them, to see whether it reacts safely to a kind of petition it has never seen. Most of
them are from the train period, because fake-name rejections almost disappeared in the current Parliament.

In [11]:
out_of_scope["split"] = assign_split(out_of_scope["decision_time"])
pd.crosstab(out_of_scope["label"], out_of_scope["split"]).reindex(columns=SPLITS, fill_value=0)

split,train,val,test,excluded
label,,,,
fake-name,829,0,0,0
foi,27,0,0,0
operational,16,3,11,0


## 6. Save

In [12]:
petitions[["id", "split"]].to_parquet(DATA / "splits.parquet", index=False)

saved = pd.read_parquet(DATA / "splits.parquet")
assert saved["id"].is_unique and len(saved) == len(petitions)
saved["split"].value_counts().reindex(SPLITS)

split
train       116417
val           4030
test          6182
excluded       562
Name: count, dtype: int64

## Summary

- **Split by decision time** (UTC): **116,417** train · **4,030** validation · **6,182** test · 562 excluded.
  The class × split table matches the locked table exactly.
- **The class mix shifts over time:** `local` is 0.2% of train but 3.7% of validation; published rises from 30% to 46–49%.
- **Retrieval queries:** **41,744** train · **1,155** validation · **1,728** test (77 more in the excluded period).
- **Candidate pools:** 40,381 published petitions can be members. They are chosen by label, so a petition that was
  published and later rejected never is one, and a petition is never in its own pool. Median open pool:
  1,929 in validation and 2,257 in test; stress pool about 38.5k at test time.
- **Gold targets:** 99.87% are in the open pool and all of them are in the stress pool. So the best Recall@k any
  method can reach is 99.83% on validation and 99.94% on test.
- **Out-of-scope petitions:** 872 from the train period, 3 from validation, 11 from test.

Next: the Colab notebook (sentence vectors + DistilRoBERTa) and `05_baseline_classifier.ipynb`.